<a href="https://colab.research.google.com/github/WVF-1/Cast-and-Crew-Analytics/blob/main/Merge_and_Parse.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎬 Cast, Crew & Keyword Intelligence — Notebook 4
## Data Cleaning & Merging

**Series:** May Newsletter — Movie Intelligence (Part 2 of 3)
**Prerequisites:** `movies_clean.parquet` from Notebook 1, plus `credits.csv` and `keywords.csv`

### What we do here
1. Load and inspect `credits.csv` and `keywords.csv`
2. Parse the nested JSON fields: extract **director**, **lead cast**, and **keyword lists**
3. Build three flat, analysis-ready tables:
   - `df_directors` — one row per film with director name attached
   - `df_actors` — one row per (film × top-billed actor) pair
   - `df_keywords` — one row per (film × keyword) pair
4. Merge all three with the financial data from NB1
5. Save parquet files for NB5 and NB6


## 0 · Imports & Setup

In [3]:
import pandas as pd
import numpy as np
import ast
import warnings
warnings.filterwarnings("ignore")

print("pandas:", pd.__version__)


pandas: 2.2.2


## 1 · Load Source Files

In [4]:
df_movies = pd.read_parquet("movies_clean.parquet")
credits   = pd.read_csv("credits.csv")
keywords  = pd.read_csv("keywords.csv")

# Harmonise id types — movies_clean stores id as string (from raw metadata)
df_movies["id"] = df_movies["id"].astype(str).str.strip()
credits["id"]   = credits["id"].astype(str).str.strip()
keywords["id"]  = keywords["id"].astype(str).str.strip()

print(f"movies_clean : {df_movies.shape}")
print(f"credits      : {credits.shape}")
print(f"keywords     : {keywords.shape}")


movies_clean : (5128, 23)
credits      : (45476, 3)
keywords     : (46419, 2)


## 2 · Parse the `crew` Column → Extract Directors

In [5]:
def extract_director(crew_raw):
    """Return the first Director found in the crew JSON list, else NaN."""
    try:
        crew = ast.literal_eval(str(crew_raw))
        for member in crew:
            if isinstance(member, dict) and member.get("job") == "Director":
                return member.get("name")
    except Exception:
        pass
    return np.nan

credits["director"] = credits["crew"].apply(extract_director)

# How many films have a director identified?
has_director = credits["director"].notna().sum()
print(f"Director extracted  : {has_director:,} / {len(credits):,} films")
print("Sample directors:")
print(credits[["id","director"]].dropna().head(8).to_string(index=False))


Director extracted  : 44,589 / 45,476 films
Sample directors:
   id        director
  862   John Lasseter
 8844    Joe Johnston
15602   Howard Deutch
31357 Forest Whitaker
11862   Charles Shyer
  949    Michael Mann
11860  Sydney Pollack
45325    Peter Hewitt


## 3 · Parse the `cast` Column → Extract Top-Billed Actors

In [6]:
def extract_top_cast(cast_raw, top_n=3):
    """Return a list of the top-N billed actor names (by billing order)."""
    try:
        cast = ast.literal_eval(str(cast_raw))
        if not isinstance(cast, list):
            return []
        sorted_cast = sorted(cast, key=lambda x: x.get("order", 999))
        return [m["name"] for m in sorted_cast[:top_n] if isinstance(m, dict) and "name" in m]
    except Exception:
        return []

def extract_lead_actor(cast_raw):
    """Return the single top-billed actor name."""
    top = extract_top_cast(cast_raw, top_n=1)
    return top[0] if top else np.nan

def extract_cast_size(cast_raw):
    """Return the total number of cast members listed."""
    try:
        cast = ast.literal_eval(str(cast_raw))
        return len(cast) if isinstance(cast, list) else 0
    except Exception:
        return 0

credits["top_cast"]   = credits["cast"].apply(lambda x: extract_top_cast(x, top_n=3))
credits["lead_actor"] = credits["cast"].apply(extract_lead_actor)
credits["cast_size"]  = credits["cast"].apply(extract_cast_size)

print(f"Lead actor extracted : {credits['lead_actor'].notna().sum():,} films")
print(f"Avg cast size        : {credits['cast_size'].mean():.1f}")
print()
print("Sample top-cast rows:")
credits[["id","director","lead_actor","cast_size"]].dropna(subset=["director"]).head(8)


Lead actor extracted : 43,058 films
Avg cast size        : 12.4

Sample top-cast rows:


,id,director,lead_actor,cast_size
0,862,John Lasseter,Tom Hanks,13
1,8844,Joe Johnston,Robin Williams,26
2,15602,Howard Deutch,Walter Matthau,7
3,31357,Forest Whitaker,Whitney Houston,10
4,11862,Charles Shyer,Steve Martin,12
5,949,Michael Mann,Al Pacino,65
6,11860,Sydney Pollack,Harrison Ford,57
7,45325,Peter Hewitt,Jonathan Taylor Thomas,7


## 4 · Parse the `keywords` Column

In [8]:
def parse_keywords(raw):
    """Return a list of keyword name strings."""
    try:
        items = ast.literal_eval(str(raw))
        return [k["name"] for k in items if isinstance(k, dict) and "name" in k]
    except Exception:
        return []

keywords["keyword_list"]  = keywords["keywords"].apply(parse_keywords)
keywords["keyword_count"] = keywords["keyword_list"].apply(len)

print(f"Avg keywords per film : {keywords['keyword_count'].mean():.1f}")
print(f"Max keywords          : {keywords['keyword_count'].max()}")

# Flatten to one row per (film, keyword)
kw_long = (
    keywords[["id","keyword_list"]]
    .explode("keyword_list")
    .rename(columns={"keyword_list": "keyword"})
    .dropna(subset=["keyword"])
    .query("keyword != ''")
)
print(f"Flat keyword table    : {len(kw_long):,} rows")
kw_long.head(8)

Avg keywords per film : 3.4
Max keywords          : 149
Flat keyword table    : 158,680 rows


,id,keyword
0,862,jealousy
0,862,toy
0,862,boy
0,862,friendship
0,862,friends
0,862,rivalry
0,862,boy next door
0,862,new toy


## 5 · Build `df_directors` — Film-Level Table with Director

In [9]:
# Merge credits (director + cast metadata) onto the clean movie financials
crew_slim = credits[["id","director","lead_actor","top_cast","cast_size"]].copy()

df_directors = df_movies.merge(crew_slim, on="id", how="inner")

print(f"df_directors shape : {df_directors.shape}")
print(f"Director coverage  : {df_directors['director'].notna().mean()*100:.1f}% of merged films")
df_directors[["title","release_year","budget","revenue","ROI","director","lead_actor","cast_size"]].head(6)


df_directors shape : (5140, 27)
Director coverage  : 100.0% of merged films


,title,release_year,budget,revenue,ROI,director,lead_actor,cast_size
0,Toy Story,1995.0,30000000.0,373554033.0,11.451801,John Lasseter,Tom Hanks,13
1,Jumanji,1995.0,65000000.0,262797249.0,3.043035,Joe Johnston,Robin Williams,26
2,Waiting to Exhale,1995.0,16000000.0,81452156.0,4.090760,Forest Whitaker,Whitney Houston,10
3,Heat,1995.0,60000000.0,187436818.0,2.123947,Michael Mann,Al Pacino,65
4,Sudden Death,1995.0,35000000.0,64350171.0,0.838576,Peter Hyams,Jean-Claude Van Damme,6
5,GoldenEye,1995.0,58000000.0,352194034.0,5.072311,Martin Campbell,Pierce Brosnan,20


## 6 · Build `df_actors` — Long Table (Film × Lead Actor)

In [10]:
# Explode top_cast so each top-3 actor gets its own row linked to the film's financials
df_actors_long = df_directors.copy()
df_actors_long = df_actors_long.explode("top_cast").rename(columns={"top_cast": "actor"})
df_actors_long = df_actors_long.dropna(subset=["actor"])

# Also build a separate lead-actor-only table for the headline leaderboard
df_lead = df_directors.dropna(subset=["lead_actor"]).copy()

print(f"df_actors_long (top-3 exploded) : {len(df_actors_long):,} rows")
print(f"df_lead (lead actor only)       : {len(df_lead):,} rows")
df_actors_long[["title","actor","ROI","revenue","primary_genre"]].head(8)


df_actors_long (top-3 exploded) : 15,390 rows
df_lead (lead actor only)       : 5,135 rows


,title,actor,ROI,revenue,primary_genre
0,Toy Story,Tom Hanks,11.451801,373554033.0,Animation
0,Toy Story,Tim Allen,11.451801,373554033.0,Animation
0,Toy Story,Don Rickles,11.451801,373554033.0,Animation
1,Jumanji,Robin Williams,3.043035,262797249.0,Adventure
1,Jumanji,Jonathan Hyde,3.043035,262797249.0,Adventure
1,Jumanji,Kirsten Dunst,3.043035,262797249.0,Adventure
2,Waiting to Exhale,Whitney Houston,4.090760,81452156.0,Comedy
2,Waiting to Exhale,Angela Bassett,4.090760,81452156.0,Comedy


## 7 · Build `df_keywords` — Long Table (Film × Keyword)

In [11]:
# Merge keyword long table onto movie financials
df_kw = kw_long.merge(
    df_movies[["id","title","release_year","revenue","ROI","vote_average","primary_genre"]],
    on="id", how="inner"
)

print(f"df_keywords shape : {df_kw.shape}")
print(f"Unique keywords   : {df_kw['keyword'].nunique():,}")
df_kw.head(6)


df_keywords shape : (43004, 8)
Unique keywords   : 10,582


,id,keyword,title,release_year,revenue,ROI,vote_average,primary_genre
0,862,jealousy,Toy Story,1995.0,373554033.0,11.451801,7.7,Animation
1,862,toy,Toy Story,1995.0,373554033.0,11.451801,7.7,Animation
2,862,boy,Toy Story,1995.0,373554033.0,11.451801,7.7,Animation
3,862,friendship,Toy Story,1995.0,373554033.0,11.451801,7.7,Animation
4,862,friends,Toy Story,1995.0,373554033.0,11.451801,7.7,Animation
5,862,rivalry,Toy Story,1995.0,373554033.0,11.451801,7.7,Animation


## 8 · Quick Data Quality Summary

In [12]:
print("=== Merge Coverage Summary ===")
print(f"  Base clean films      : {len(df_movies):,}")
print(f"  After credits merge   : {len(df_directors):,}  ({len(df_directors)/len(df_movies)*100:.1f}%)")
print(f"  With director known   : {df_directors['director'].notna().sum():,}")
print(f"  With lead actor known : {df_directors['lead_actor'].notna().sum():,}")
print(f"  With keyword data     : {df_kw['id'].nunique():,}")
print()
print("=== Director Statistics ===")
dir_counts = df_directors.dropna(subset=["director"])["director"].value_counts()
print(f"  Unique directors      : {dir_counts.shape[0]:,}")
print(f"  Most prolific         : {dir_counts.index[0]}  ({dir_counts.iloc[0]} films)")
print()
print("=== Actor Statistics ===")
actor_counts = df_actors_long["actor"].value_counts()
print(f"  Unique actors (top-3) : {actor_counts.shape[0]:,}")
print(f"  Most frequent         : {actor_counts.index[0]}  ({actor_counts.iloc[0]} appearances)")
print()
print("=== Keyword Statistics ===")
kw_counts = df_kw["keyword"].value_counts()
print(f"  Unique keywords       : {kw_counts.shape[0]:,}")
print(f"  Most common           : {kw_counts.index[0]}  ({kw_counts.iloc[0]} films)")


=== Merge Coverage Summary ===
  Base clean films      : 5,128
  After credits merge   : 5,140  (100.2%)
  With director known   : 5,140
  With lead actor known : 5,135
  With keyword data     : 4,887

=== Director Statistics ===
  Unique directors      : 2,225
  Most prolific         : Steven Spielberg  (30 films)

=== Actor Statistics ===
  Unique actors (top-3) : 5,732
  Most frequent         : Robert De Niro  (49 appearances)

=== Keyword Statistics ===
  Unique keywords       : 10,582
  Most common           : duringcreditsstinger  (315 films)


## 9 · Save All Tables

In [13]:
df_directors.to_parquet("directors_clean.parquet", index=False)
df_actors_long.to_parquet("actors_long.parquet",   index=False)
df_lead.to_parquet("lead_actors.parquet",           index=False)
df_kw.to_parquet("keywords_clean.parquet",          index=False)

print("Saved:")
print("  ✔  directors_clean.parquet")
print("  ✔  actors_long.parquet")
print("  ✔  lead_actors.parquet")
print("  ✔  keywords_clean.parquet")
print()
print("Proceed to Notebook 5 → Creative Intelligence Analysis ▶")


Saved:
  ✔  directors_clean.parquet
  ✔  actors_long.parquet
  ✔  lead_actors.parquet
  ✔  keywords_clean.parquet

Proceed to Notebook 5 → Creative Intelligence Analysis ▶
